# Spark SQL: Temporary Views, Global Views, and SQL Execution

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 04_spark_sql.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Writing declarative ANSI SQL queries against Spark DataFrames, managing temporary and global views, and using Common Table Expressions (CTEs).
- **Why does this topic exist?**: SQL is the universal language of data analytics. Spark SQL compiles ANSI SQL into the exact same optimized Catalyst physical plans as the DataFrame API.
- **What problem does it solve?**: Knowing when to use raw SQL versus programmatic DataFrame transformations.
- **Where is it used in real-world Data Engineering?**: Migrating legacy SQL warehouse queries (Snowflake, BigQuery, Oracle) to Apache Spark.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DataFrames, schemas |
| **Required depth** | Working knowledge: can use views, the catalog, and SQL plans |
| **Data Engineer** | Write Spark SQL, manage temp/global views, and explain the shared execution engine. |
| **Senior Data Engineer** | Choose SQL vs DataFrame based on maintainability, governance, and performance. |
| **Spark specialist** | Reason about the parser, analyzer, catalog, and SQL configuration. |
| **Next topic** | `03_dataframe_and_spark_sql/05_joins_and_aggregations.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Spark SQL is not a separate engine; it compiles to the same Catalyst plans as the DataFrame API. That means
the choice between SQL and DataFrames is mostly about readability, team skills, and governance, not raw speed.
Views are metadata aliases, not copies of data, and their scope matters in notebooks and shared sessions.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Temp view** | Session-scoped table alias | `createOrReplaceTempView` |
| **Global temp view** | Application-scoped alias | Accessible in the `global_temp` database |
| **Catalog** | Metadata about tables/views | Session catalog by default in this lab |
| **SQL plan** | Parsed/analyzed/optimized plan | Same Catalyst pipeline as DataFrames |
| **ANSI mode** | Strict SQL semantics | Invalid casts/operations can fail |
| **View vs table** | Alias vs persisted metadata/data | A view stores a query; a table stores data/metadata |

### How it works, step by step

1. SQL text is parsed into an unresolved logical plan.
2. The analyzer resolves tables, columns, and functions using the catalog.
3. Optimizer rules rewrite the plan.
4. The physical planner selects operators and join strategies.
5. The driver schedules the resulting job on executors.
6. Views are resolved at query time; they do not materialize data unless cached.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Spark SQL vs DataFrame API Equivalence

```
User ANSI SQL Query:
"SELECT region, sum(revenue) FROM sales GROUP BY region"
                      |
                      v (Parsed by Catalyst SQL Parser)
          Parsed Logical Plan
                      ^
                      | (Built by DataFrame API)
User DataFrame API Call:
df.groupBy("region").agg(sum("revenue"))
```
> Both APIs compile into the **exact same Physical Plan and execute identically**!

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_SQL_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Registering Views and Running SQL Queries

In [ ]:
# Register retail datasets as SQL views
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
customers = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/customers.csv")

orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

# Query with CTE and aggregations
sql_query = """
WITH customer_revenue AS (
    SELECT
        o.customer_id,
        c.region,
        c.segment,
        SUM(o.quantity * o.unit_price) AS total_spent,
        COUNT(o.order_id) AS total_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'Completed'
    GROUP BY o.customer_id, c.region, c.segment
)
SELECT
    region,
    segment,
    COUNT(customer_id) AS active_customers,
    ROUND(SUM(total_spent), 2) AS regional_revenue,
    ROUND(AVG(total_spent), 2) AS avg_customer_spend
FROM customer_revenue
GROUP BY region, segment
ORDER BY regional_revenue DESC
"""

result_df = spark.sql(sql_query)
result_df.show(10)

---

## Experiment: Verifying Physical Plan Parity: SQL vs DataFrame API

- **1. What are we testing?**: Prove that an identical SQL query and DataFrame call generate the exact same Catalyst physical plan.
- **2. What do we expect?**: Both explain() outputs will show identical physical operators.

In [ ]:
# SQL approach
plan_sql = spark.sql("SELECT status, count(*) as cnt FROM orders GROUP BY status")

# DataFrame approach
plan_df = orders.groupBy("status").count()

print("=== SQL Physical Plan ===")
plan_sql.explain()

print("\n=== DataFrame Physical Plan ===")
plan_df.explain()

**3. What happened?**: Both approaches generated identical HashAggregate and Exchange physical operators.  
**4. Why did it happen?**: Spark SQL parses SQL into the same Catalyst AST (Abstract Syntax Tree) that DataFrame methods build.  
**5. Practical Engineering Lesson**: Choose SQL or DataFrame API based on readability and team preference; performance is identical.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: BI reporting layer over a lake

- **Industry**: Retail
- **Scenario**: Analysts write SQL against curated order and product views.
- **Problem**: The data team cannot build a DataFrame for every question, and raw file paths are error-prone.
- **Why this concept is relevant**: Spark SQL views provide a familiar interface over the same distributed engine.
- **Implementation idea**: Create curated temp/global views with documented columns, grant access through the platform, and monitor query plans.
- **Expected result**: Faster self-service reporting with a shared semantic layer.
- **Alternative**: A warehouse with a BI semantic layer.
- **Trade-offs**: Spark SQL scales over lake data; a warehouse offers better concurrency and governance for many BI users.

### Use Case 2: Migrating warehouse SQL to a lakehouse

- **Industry**: Banking
- **Scenario**: A bank moves some reporting SQL from a warehouse to a lakehouse.
- **Problem**: Queries must remain readable and auditable while the storage layer changes.
- **Why this concept is relevant**: Spark SQL preserves the SQL interface and uses Catalyst for distributed execution.
- **Implementation idea**: Port queries incrementally, compare results with the warehouse, and validate plans and data types.
- **Expected result**: A validated migration with side-by-side reconciliation.
- **Alternative**: A lakehouse query engine with a warehouse-compatible SQL dialect.
- **Trade-offs**: Spark SQL is flexible; a dedicated lakehouse engine may offer better concurrency and caching.

---

## Business Value

- Lowers the barrier for analysts and SQL-literate engineers.
- Keeps one execution engine for SQL and DataFrames.
- Makes curated views a lightweight semantic layer.
- Can reduce duplicate logic between teams.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use schema-qualified names and avoid relying on session state.
- Document view definitions and ownership.
- Parameterize queries; do not concatenate untrusted input.
- Check ANSI behavior when migrating SQL.
- Monitor query plans for accidental cross joins.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: View scope and catalog

**Steps**

1. Create a temp view and query it with `spark.sql`.
2. Inspect `spark.catalog.listTables()`.
3. Create a global temp view and query `global_temp.<name>`.
4. Drop the views and confirm they disappear.

**What to observe**: Temp views are session-scoped; global temp views are application-scoped.

**Healthy result**: You can explain which users/sessions can see each view.

**Problematic result**: A view created in one session is not visible in another.

**Engineering takeaway**: Views are metadata aliases with scope, not data copies.

---

## Trade-offs and Alternatives

### Advantages

- Familiar interface
- Same optimizer as DataFrames
- Easy ad-hoc analysis

### Limitations

- String SQL is harder to refactor/test
- View scope can confuse users
- Dialect differences during migration

### When to use it

- BI/reporting
- Ad-hoc analysis
- SQL migrations

### When not to use it

- Complex procedural logic
- Unparameterized user input
- Assuming a view is materialized

### Alternatives inside Spark

- DataFrame API
- `selectExpr`
- Catalog tables

### Alternatives outside Spark

- Warehouse SQL
- dbt
- Trino/Presto

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Monthly Revenue Trend Query
**Problem Statement**: Write an ANSI SQL query to compute monthly completed revenue and order count for 2025.  
**Input / Context**: orders view

In [ ]:
# Exercise 1: Monthly revenue in SQL

### Exercise 2 (Intermediate): Top 5 Products per Channel Query
**Problem Statement**: Write a SQL query using window function DENSE_RANK() to find the top 3 selling products per channel.  
**Input / Context**: orders view

In [ ]:
# Exercise 2: Top products per channel

### Exercise 3 (Advanced): Multi-Table SQL Join with Customer Loyalty
**Problem Statement**: Query revenue by loyalty_tier and channel, showing percentage share of total revenue.  
**Input / Context**: orders and customers views

In [ ]:
# Exercise 3: Revenue share by loyalty

---

## Production Debugging Exercise

### View not found

**Scenario**: A query fails with `TABLE_OR_VIEW_NOT_FOUND` in a new session.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the view was temporary and session-scoped.
2. List the catalog tables.
3. Use a persisted table or recreate the view in the new session.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Monthly Revenue Trend Query):
```python
spark.sql("""
    SELECT 
        date_format(order_ts,'yyyy-MM') AS order_month,
        COUNT(order_id) AS total_orders,
        ROUND(SUM(quantity * unit_price), 2) AS monthly_revenue
    FROM orders
    WHERE status ='Completed'
    GROUP BY date_format(order_ts,'yyyy-MM')
    ORDER BY order_month
""").show()
```

#### Solution 2 (Top 5 Products per Channel Query):
```python
spark.sql("""
    WITH ranked_products AS (
        SELECT 
            channel,
            product_id,
            SUM(quantity) AS total_units,
            DENSE_RANK() OVER (PARTITION BY channel ORDER BY SUM(quantity) DESC) as rnk
        FROM orders
        GROUP BY channel, product_id
)
    SELECT * FROM ranked_products WHERE rnk <= 3 ORDER BY channel, rnk
""").show()
```

#### Solution 3 (Multi-Table SQL Join with Customer Loyalty):
```python
spark.sql("""
    SELECT 
        c.loyalty_tier,
        o.channel,
        ROUND(SUM(o.quantity * o.unit_price), 2) AS tier_revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.loyalty_tier, o.channel
    ORDER BY tier_revenue DESC
""").show(10)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **SQL vs DataFrame**: Identical execution plans and performance under Catalyst.
- **Temp Views**: Session-scoped virtual tables registered in internal Catalog.
- **Advanced SQL**: Supports full ANSI SQL including CTEs, Window functions, and nested queries.

### Essential Technical Terms
- `createOrReplaceTempView`: Registers a DataFrame as a temporary SQL view.
- `spark.sql()`: Executes an SQL query string and returns a DataFrame.
- `CTE`: Common Table Expression (WITH clause).

### Common Engineering Mistakes
- **Assuming SQL is slower than DataFrames**: False: both compile to the exact same bytecode.
- **SQL Injection in Unsanitized Strings**: Avoid dynamic f-string queries with untrusted user input.

### Interview Questions on This Topic
- **Q: Is Spark SQL slower than the DataFrame API?**
  - *A*: No, both produce identical Catalyst logical plans and compile into the same Tungsten physical bytecode.
- **Q: How long do temporary views last?**
  - *A*: Only for the lifetime of that specific SparkSession.


### More Interview Questions

- **Q: Is Spark SQL a separate engine from DataFrames?**
  - *A*: No. Both use the same Spark SQL engine and Catalyst optimizer; SQL is parsed into a logical plan.
- **Q: What is the difference between a temp view and a global temp view?**
  - *A*: A temp view is visible only in the current session; a global temp view is visible across sessions in the same application and lives in the `global_temp` database.

### What to Study Next
Open 03_dataframe_and_spark_sql/05_joins_and_aggregations.ipynb to master distributed joins.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')